# 이벤트 스키마 설계 및 합성 데이터 생성

## 분석 목적
Amazon 리뷰 데이터의 상품 품질 점수를 콘텐츠 품질 proxy로 활용해
사용자 engagement 패턴을 분석하기 위한 기반 데이터 생성

## 핵심 가설
- 콘텐츠 품질이 높을수록 completion_rate가 높다.
- 첫 세션에서 고품질 콘텐츠에 노출된 사용자일수록 D7 retention이 높다
- 추천 다양성이 낮을수록 D30 retention이 낮다

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import platform

# 한글 폰트 설정
if platform.system() == 'Darwin':
    matplotlib.rc('font', family='AppleGothic')
matplotlib.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

df = pd.read_csv('amazon_cleaned.csv')
print(df.shape)
df.head()

(1212, 22)


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,...,review_title,review_content,img_link,product_link,discount_amount,price_tier,discount_tier,rating_tier,category_main,category_sub
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,399.0,1099.0,64,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...",...,"Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...,700.0,~500,61~80%,4.0~4.5,Computers&Accessories,Accessories&Peripherals
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,199.0,349.0,43,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...",...,"A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...,150.0,~500,41~60%,3.5~4.0,Computers&Accessories,Accessories&Peripherals
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,199.0,1899.0,90,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...",...,"Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...,1700.0,~500,81~100%,3.5~4.0,Computers&Accessories,Accessories&Peripherals
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,329.0,699.0,53,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...",...,"Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...,370.0,~500,41~60%,4.0~4.5,Computers&Accessories,Accessories&Peripherals
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,154.0,399.0,61,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...",...,"As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...,245.0,~500,61~80%,4.0~4.5,Computers&Accessories,Accessories&Peripherals


## 콘텐츠 품질 점수(content_quality_score) 생성

In [2]:
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['rating_count'] = pd.to_numeric(df['rating_count'].astype(str).str.replace(',', ''), errors='coerce')

df = df.dropna(subset=['rating', 'rating_count'])

df['rating_norm'] = (df['rating'] - df['rating'].min()) / (df['rating'].max() - df['rating'].min())
df['count_norm'] = (df['rating_count'] - df['rating_count'].min()) / (df['rating_count'].max() - df['rating_count'].min())

df['content_quality_score'] = (df['rating_norm'] * 0.7) + (df['count_norm'] * 0.3)

print(df['content_quality_score'].describe())

count    1210.000000
mean        0.496614
std         0.076826
min         0.000000
25%         0.458350
50%         0.500347
75%         0.542363
max         0.860000
Name: content_quality_score, dtype: float64


## 사용자 데이터 생성

- Power User (10%): 자주 방문, 높은 engagement
- Casual User (35%): 가끔 방문, 보통 engagement  
- At-risk User (25%): 방문 감소 중, 이탈 위험
- Churned User (30%): 7일 이상 미접속

In [3]:
N_USERS = 1000

segments = np.random.choice(
    ['power', 'casual', 'at_risk', 'churned'],
    size=N_USERS,
    p=[0.10, 0.35, 0.25, 0.30]
)

df_users = pd.DataFrame({
    'user_id': [f'USR_{i:04d}' for i in range(N_USERS)],
    'segment': segments,
    'join_date': pd.to_datetime(
        np.random.choice(
            pd.date_range('2023-01-01', '2023-12-31'), 
            size=N_USERS
        )
    )
})

print(df_users['segment'].value_counts())
print()
print(df_users.head())

segment
casual     353
churned    288
at_risk    251
power      108
Name: count, dtype: int64

    user_id  segment  join_date
0  USR_0000   casual 2023-10-30
1  USR_0001  churned 2023-01-12
2  USR_0002  churned 2023-11-14
3  USR_0003  at_risk 2023-03-21
4  USR_0004   casual 2023-08-04


## 이벤트 로그 생성

세그먼트별 특징:
- power : completion_rate 높음, skip 적음, 자주 재방문
- casual : completion_rate 보통, 가끔 skip
- at_risk : completion_rate 낮아지는 중, skip 많음
- churned : 마지막 세션 이후 미접속

In [4]:
segment_behavior = {
    'power'  : {'n_sessions': (10, 20), 'completion_mean': 0.80, 'skip_prob': 0.10},
    'casual' : {'n_sessions': (3, 8),   'completion_mean': 0.55, 'skip_prob': 0.30},
    'at_risk': {'n_sessions': (1, 4),   'completion_mean': 0.35, 'skip_prob': 0.55},
    'churned': {'n_sessions': (1, 2),   'completion_mean': 0.20, 'skip_prob': 0.75},
}

content_list = df['product_id'].unique()
quality_scores = df.drop_duplicates('product_id').set_index('product_id')['content_quality_score']

segment_weight = {
    'power'  : quality_scores,
    'casual' : None,
    'at_risk': 1 - quality_scores,
    'churned': 1 - quality_scores,
}

events = []

for _, user in df_users.iterrows():
    behavior = segment_behavior[user['segment']]
    n_sessions = np.random.randint(*behavior['n_sessions'])
    
    for session in range(n_sessions):
        session_id = f"SES_{user['user_id']}_{session:02d}"
        session_date = user['join_date'] + pd.Timedelta(days=np.random.randint(0, 60))
        n_contents = np.random.randint(3, 10)
        
        weights = segment_weight[user['segment']]
        if weights is None:
            probs = None
        else:
            weights = weights.reindex(content_list).fillna(0)
            probs = (weights / weights.sum()).values
        
        contents_watched = np.random.choice(content_list, size=n_contents, replace=False, p=probs)
        
        for content_id in contents_watched:
            quality = df[df['product_id'] == content_id]['content_quality_score'].values[0]
            completion = np.clip(np.random.normal(behavior['completion_mean'] + (quality - 0.5) * 0.3, 0.15), 0, 1)
            is_skipped = np.random.random() < behavior['skip_prob']
            
            events.append({
                'user_id'               : user['user_id'],
                'segment'               : user['segment'],
                'session_id'            : session_id,
                'content_id'            : content_id,
                'event_date'            : session_date,
                'content_quality_score' : round(quality, 4),
                'completion_rate'       : round(completion, 4),
                'is_skipped'            : is_skipped,
                'watch_time_sec'        : round(completion * 30, 1),
            })

df_events = pd.DataFrame(events)
print(df_events.shape)
print(df_events.head())

(25102, 9)
    user_id segment       session_id  content_id event_date  \
0  USR_0000  casual  SES_USR_0000_00  B0BN2576GQ 2023-12-02   
1  USR_0000  casual  SES_USR_0000_00  B01M69WCZ6 2023-12-02   
2  USR_0000  casual  SES_USR_0000_00  B01EJ5MM5M 2023-12-02   
3  USR_0000  casual  SES_USR_0000_00  B08MV82R99 2023-12-02   
4  USR_0000  casual  SES_USR_0000_01  B07LDPLSZC 2023-12-27   

   content_quality_score  completion_rate  is_skipped  watch_time_sec  
0                 0.3967           0.5828       False            17.5  
1                 0.4695           0.8736        True            26.2  
2                 0.3352           0.3379       False            10.1  
3                 0.4924           0.3953        True            11.9  
4                 0.5163           0.5280       False            15.8  


In [5]:
df_events.to_csv('events.csv', index=False)
df_users.to_csv('users.csv', index=False)
print("저장 완료!")

저장 완료!
